# 02 CNN Training for Eye State Detection

This notebook trains a Convolutional Neural Network to classify eye images into:
- `open`
- `closed`

It is designed to work with the samples collected from `src/main.py`.

Expected folder structure:

```text
DriverAttentionMonitoring/
  data/
    open/
    closed/
  models/
    eye_classifier.h5
```


## Step 1: Import libraries and set paths

In [ ]:
from pathlib import Path
from typing import List, Tuple
import random

import cv2
import keras
import matplotlib.pyplot as plt
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'data'
MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_PATH = MODEL_DIR / 'eye_classifier.h5'

OPEN_DIR = DATA_DIR / 'open'
CLOSED_DIR = DATA_DIR / 'closed'

IMAGE_HEIGHT = 48
IMAGE_WIDTH = 96
IMAGE_SIZE = (IMAGE_HEIGHT, IMAGE_WIDTH)
BATCH_SIZE = 32
EPOCHS = 15

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Open folder:', OPEN_DIR)
print('Closed folder:', CLOSED_DIR)
print('Model save path:', MODEL_PATH)


## Step 2: Check dataset availability

In [ ]:
open_files = [path for path in OPEN_DIR.glob('*') if path.suffix.lower() in {'.png', '.jpg', '.jpeg'}]
closed_files = [path for path in CLOSED_DIR.glob('*') if path.suffix.lower() in {'.png', '.jpg', '.jpeg'}]

print('Open samples:', len(open_files))
print('Closed samples:', len(closed_files))

if len(open_files) == 0 or len(closed_files) == 0:
    raise RuntimeError(
        'Dataset is empty. Run src/main.py and press O for open-eye samples and C for closed-eye samples before training.'
    )


## Step 3: Load and preprocess images

In [ ]:
def load_eye_image(path: Path, image_size: Tuple[int, int]) -> np.ndarray:
    image = cv2.imread(str(path))
    if image is None:
        raise ValueError(f'Could not read image: {path}')

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (image_size[1], image_size[0]))
    normalized = resized.astype('float32') / 255.0
    return normalized


def build_dataset() -> Tuple[np.ndarray, np.ndarray, List[Path]]:
    images: List[np.ndarray] = []
    labels: List[int] = []
    file_paths: List[Path] = []

    for path in sorted(open_files):
        images.append(load_eye_image(path, IMAGE_SIZE))
        labels.append(1)
        file_paths.append(path)

    for path in sorted(closed_files):
        images.append(load_eye_image(path, IMAGE_SIZE))
        labels.append(0)
        file_paths.append(path)

    x_data = np.array(images, dtype=np.float32)[..., np.newaxis]
    y_data = np.array(labels, dtype=np.int32)
    return x_data, y_data, file_paths


x_data, y_data, file_paths = build_dataset()
print('X shape:', x_data.shape)
print('Y shape:', y_data.shape)
print('Open count:', int(np.sum(y_data == 1)))
print('Closed count:', int(np.sum(y_data == 0)))

def stratified_split_indices(labels: np.ndarray, test_size: float, seed: int) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    train_indices: List[np.ndarray] = []
    test_indices: List[np.ndarray] = []
    for label in np.unique(labels):
        label_indices = np.where(labels == label)[0]
        rng.shuffle(label_indices)
        test_count = max(1, int(round(len(label_indices) * test_size)))
        test_indices.append(label_indices[:test_count])
        train_indices.append(label_indices[test_count:])
    train_idx = np.concatenate(train_indices)
    test_idx = np.concatenate(test_indices)
    rng.shuffle(train_idx)
    rng.shuffle(test_idx)
    return train_idx, test_idx


def classification_report_text(y_true: np.ndarray, y_pred: np.ndarray) -> str:
    lines = ['label      precision  recall  f1-score  support']
    for label_value, label_name in [(0, 'closed'), (1, 'open')]:
        true_positive = int(np.sum((y_true == label_value) & (y_pred == label_value)))
        false_positive = int(np.sum((y_true != label_value) & (y_pred == label_value)))
        false_negative = int(np.sum((y_true == label_value) & (y_pred != label_value)))
        support = int(np.sum(y_true == label_value))
        precision = true_positive / max(true_positive + false_positive, 1)
        recall = true_positive / max(true_positive + false_negative, 1)
        f1_score = 2 * precision * recall / max(precision + recall, 1e-8)
        lines.append(f'{label_name:<10}{precision:>8.3f}{recall:>8.3f}{f1_score:>10.3f}{support:>9}')
    return '\n'.join(lines)


def confusion_matrix_array(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    matrix = np.zeros((2, 2), dtype=np.int32)
    for true_label, pred_label in zip(y_true, y_pred):
        matrix[int(true_label), int(pred_label)] += 1
    return matrix


## Step 4: Preview sample images

In [ ]:
def preview_samples(images: np.ndarray, labels: np.ndarray, sample_count: int = 8) -> None:
    sample_count = min(sample_count, len(images))
    indices = np.random.choice(len(images), size=sample_count, replace=False)
    rows = 2
    cols = max(1, sample_count // 2)
    fig, axes = plt.subplots(rows, cols, figsize=(14, 6))
    axes = np.array(axes).reshape(-1)

    for axis, index in zip(axes, indices):
        axis.imshow(images[index].squeeze(), cmap='gray')
        axis.set_title('open' if labels[index] == 1 else 'closed')
        axis.axis('off')

    for axis in axes[len(indices):]:
        axis.axis('off')

    plt.tight_layout()
    plt.show()


preview_samples(x_data, y_data)


## Step 5: Split the dataset

In [ ]:
train_idx, temp_idx = stratified_split_indices(y_data, test_size=0.30, seed=SEED)
x_train, y_train = x_data[train_idx], y_data[train_idx]
x_temp, y_temp = x_data[temp_idx], y_data[temp_idx]

val_idx, test_idx = stratified_split_indices(y_temp, test_size=0.50, seed=SEED + 1)
x_val, y_val = x_temp[val_idx], y_temp[val_idx]
x_test, y_test = x_temp[test_idx], y_temp[test_idx]

print('Train:', x_train.shape, y_train.shape)
print('Validation:', x_val.shape, y_val.shape)
print('Test:', x_test.shape, y_test.shape)


## Step 6: Build the CNN model

In [ ]:
augmentation = keras.Sequential([
    keras.layers.RandomFlip('horizontal'),
    keras.layers.RandomRotation(0.05),
    keras.layers.RandomZoom(0.05),
], name='augmentation')


def build_model(input_shape: Tuple[int, int, int]) -> keras.Model:
    inputs = keras.Input(shape=input_shape)
    x = augmentation(inputs)
    x = keras.layers.Conv2D(16, (3, 3), activation='relu', padding='same')(x)
    x = keras.layers.MaxPooling2D()(x)
    x = keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = keras.layers.MaxPooling2D()(x)
    x = keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = keras.layers.MaxPooling2D()(x)
    x = keras.layers.Flatten()(x)
    x = keras.layers.Dense(128, activation='relu')(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(1, activation='sigmoid')(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name='eye_state_classifier')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')],
    )
    return model


model = build_model((IMAGE_HEIGHT, IMAGE_WIDTH, 1))
model.summary()


## Step 7: Train the model

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(filepath=str(MODEL_PATH), monitor='val_accuracy', save_best_only=True),
]

history = model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)


## Step 8: Visualize training history

In [ ]:
history_data = history.history
epochs_ran = range(1, len(history_data['loss']) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_ran, history_data['accuracy'], label='train')
plt.plot(epochs_ran, history_data['val_accuracy'], label='validation')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_ran, history_data['loss'], label='train')
plt.plot(epochs_ran, history_data['val_loss'], label='validation')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()


## Step 9: Evaluate on test data

In [ ]:
test_loss, test_accuracy, test_precision, test_recall = model.evaluate(x_test, y_test, verbose=0)
print('Test loss:', round(float(test_loss), 4))
print('Test accuracy:', round(float(test_accuracy), 4))
print('Test precision:', round(float(test_precision), 4))
print('Test recall:', round(float(test_recall), 4))

y_prob = model.predict(x_test, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(np.int32)

print('\nClassification report:')
print(classification_report_text(y_test, y_pred))

print('Confusion matrix:')
print(confusion_matrix_array(y_test, y_pred))


## Step 10: Save the final model

In [ ]:
model.save(MODEL_PATH)
print('Saved model to:', MODEL_PATH)
